## Support Displacement of a Frame


### Solve Question 7.3 in Kassimali

Determine the joint displacements, member end forces, and support reactions for a 
continuous beam, due to the combined effect of the uniformly distributed load shown and the settlements of 45 mm and 15 mm, respectively, of supports 3 and 4. 

Using the DSM, model the structure as frame elements, not beam elements, you should get the same answer as the textbook.


<div style="text-align:center;">
  <img src="../../Lectures/L9/assets/L1_Example2.png" style="width:95%;">
</div>

In [19]:
# -------------------------
# Import Libraries and Helper Functions
# -------------------------
# This slide imports the required Python libraries and adds the
# course helper-code directory to the Python path.

import sys
import os
import numpy as np
import pandas as pd

# Get current notebook directory
current_dir = os.getcwd()

# Go up two levels, then into Code/L8
helpers_path = os.path.abspath(
    os.path.join(current_dir, "..", "..", "Code", "L8")
)

sys.path.append(helpers_path)

In [20]:
import numpy as np

from helper_funcs_dsm import (
    assemble_global_stiffness_and_fef,
    partition_system,
    assemble_global_displacements,
    assemble_global_forces
)
from helper_funcs_output import (
    print_element,
    print_dsm_results,
    print_matrix_scaled
)


In [21]:
# -------------------------
# Problem data (hard-coded)
# -------------------------
E = 200e6          # kPa
# A = 0.0065         # m^2
I = 102e-6         # m^4
L = 8.0            # m

In [22]:
# -------------------------
# Member 1
# -------------------------
k1 = np.array([
    [162500.,     0.,       0.,  -162500.,     0.,       0.],
    [    0.,   167.34,   669.38,      0.,  -167.34,   669.38],
    [    0.,   669.38,  3570.00,      0.,  -669.38,  1785.00],
    [-162500.,    0.,       0.,   162500.,     0.,       0.],
    [    0.,  -167.34,  -669.38,      0.,   167.34,  -669.38],
    [    0.,   669.38,  1785.00,      0.,  -669.38,  3570.00]
], dtype=float)

# Transformation matrix for theta=0 (identity matrix)
T1 = np.eye(6, dtype=float)
# Mapping
m1 = np.array([1,2,3,4,5,6])

In [23]:
# Local fixed-end vector for member 1 (given by the example result)
Qf1 = np.array([0., 60., 80., 0., 60., -80.], dtype=float)

In [24]:
# -------------------------
# Member 2
# -------------------------
k2 = np.array([
    [162500.,     0.,       0.,  -162500.,     0.,       0.],
    [    0.,   167.34,   669.38,      0.,  -167.34,   669.38],
    [    0.,   669.38,  3570.00,      0.,  -669.38,  1785.00],
    [-162500.,    0.,       0.,   162500.,     0.,       0.],
    [    0.,  -167.34,  -669.38,      0.,   167.34,  -669.38],
    [    0.,   669.38,  1785.00,      0.,  -669.38,  3570.00]
], dtype=float)

# Transformation matrix for theta=0 (identity matrix)
T2 = np.eye(6, dtype=float)
# Mapping
m2 = np.array([4,5,6,7,8,9])

In [25]:
# Local fixed-end vector for member 1 (given by the example result)
Qf2 = np.array([0., 60., 80., 0., 60., -80.], dtype=float)

In [26]:
# -------------------------
# Member 3
# -------------------------
k3 = np.array([
    [162500.,     0.,       0.,  -162500.,     0.,       0.],
    [    0.,   167.34,   669.38,      0.,  -167.34,   669.38],
    [    0.,   669.38,  3570.00,      0.,  -669.38,  1785.00],
    [-162500.,    0.,       0.,   162500.,     0.,       0.],
    [    0.,  -167.34,  -669.38,      0.,   167.34,  -669.38],
    [    0.,   669.38,  1785.00,      0.,  -669.38,  3570.00]
], dtype=float)

# Transformation matrix for theta=0 (identity matrix)
T3 = np.eye(6, dtype=float)
# Mapping
m3 = np.array([7,8,9,10,11,12])

In [27]:
# Local fixed-end vector for member 1 (given by the example result)
Qf3 = np.array([0., 60., 80., 0., 60., -80.], dtype=float)

In [28]:
# -------------------------
# Initialize Global System
# -------------------------
# This slide sets up the global variables needed for assembly of the
# Direct Stiffness Method system.

k_list = [k1, k2, k3]
T_list = [T1, T2, T3]
Qf_list = [Qf1, Qf2, Qf3]
map_list = [m1, m2, m3]   # 1-based DOF maps

# Global system size (still using 1-based maps here)
ndof = int(np.max(np.concatenate(map_list)))

# Initialize Applied Load
F_global = np.zeros(ndof, dtype=float)

# Initialize Global Displacement
u_global = np.zeros(ndof, dtype=float)

In [29]:
# -------------------------
# User-Specified Boundary Conditions
# -------------------------
# This slide defines the external loads and support conditions.

# Applied Load
# F_global[0] = 0.0

# Global Displacement (0-indexed)
u_global[8-1] = -0.045 #m
u_global[11-1] = -0.015 #m

# Restrained DOFs
dof_restrained_1based = np.array([1, 2, 3, 5, 8, 11], dtype=int)

# # Fictitious restrained DOFs
# dof_fictitious_1based = np.array([], dtype=int)

# # Combine and sort
# dof_restrained_1based = np.sort(
#     np.concatenate((dof_restrained_1based, dof_fictitious_1based))
# )

In [30]:
# -------------------------
# Assemble Global Stiffness System
# -------------------------
# This step assembles the global stiffness matrix and the global
# fixed-end force vector from all elements.

K_global, F_fef_global = assemble_global_stiffness_and_fef(
    ndof,
    k_list,
    T_list,
    Qf_list,
    map_list
    )

In [31]:
# -------------------------
# Display Global Stiffness Matrix
# -------------------------
# Print the assembled global stiffness matrix in a readable format.
# The matrix is scaled for clearer presentation in the slides.

print_matrix_scaled(K_global, scale=1000, decimals=2, col_width=6)

K = 1e+03 ×
01 | 162.50   0.00   0.00 -162.50   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00
02 |   0.00   0.17   0.67   0.00  -0.17   0.67   0.00   0.00   0.00   0.00   0.00   0.00
03 |   0.00   0.67   3.57   0.00  -0.67   1.78   0.00   0.00   0.00   0.00   0.00   0.00
04 | -162.50   0.00   0.00 325.00   0.00   0.00 -162.50   0.00   0.00   0.00   0.00   0.00
05 |   0.00  -0.17  -0.67   0.00   0.33   0.00   0.00  -0.17   0.67   0.00   0.00   0.00
06 |   0.00   0.67   1.78   0.00   0.00   7.14   0.00  -0.67   1.78   0.00   0.00   0.00
07 |   0.00   0.00   0.00 -162.50   0.00   0.00 325.00   0.00   0.00 -162.50   0.00   0.00
08 |   0.00   0.00   0.00   0.00  -0.17  -0.67   0.00   0.33   0.00   0.00  -0.17   0.67
09 |   0.00   0.00   0.00   0.00   0.67   1.78   0.00   0.00   7.14   0.00  -0.67   1.78
10 |   0.00   0.00   0.00   0.00   0.00   0.00 -162.50   0.00   0.00 162.50   0.00   0.00
11 |   0.00   0.00   0.00   0.00   0.00   0.00   0.00  -0.17  -0.67   0.00   0.17  -0.67
12 

In [32]:
# -------------------------
# Partition Global System
# -------------------------
# The global stiffness equation is partitioned into free and restrained
# degrees of freedom based on the support conditions.

(
    K_ff,
    K_fr,
    K_rf,
    K_rr,
    f_f,
    f_r,
    u_r,
    f_fef_f,
    f_fef_r,
    free_dofs,
    restrained_dofs,
) = partition_system(
    K_global,
    F_global,
    u_global,
    F_fef_global,
    dof_restrained_1based
)

print(u_r)


[ 0.     0.     0.     0.    -0.045 -0.015]


In [33]:
# -------------------------
# Solve Partitioned System
# -------------------------
# Solve for the unknown displacements at the free DOFs,
# compute the reactions at restrained DOFs, and then
# reconstruct the complete global displacement and force vectors.

rhs = f_f - f_fef_f - K_fr @ u_r
u_f = np.linalg.solve(K_ff, rhs)

F_r = K_rf @ u_f + K_rr @ u_r + f_fef_r

u_global = assemble_global_displacements(
    u_f,
    u_r,
    free_dofs,
    restrained_dofs
    )

f_global_complete = assemble_global_forces(
    f_f,
    F_r,
    free_dofs,
    restrained_dofs
    )

In [34]:
# -------------------------
# Display DSM Results
# -------------------------
# Print the final global displacement vector and force vector.

print_dsm_results(
    u_global,
    f_global_complete,
    dof_restrained_1based,
    disp_in_mm=True
)

 DOF  Type Status  Disp (mm / rad)  Load (kN / kN·m)
   1   u_x  Fixed            0.000             0.000
   2   u_y  Fixed            0.000            58.692
   3 theta  Fixed            0.000            76.512
   4   u_x   Free            0.000             0.000
   5   u_y  Fixed            0.000           121.467
   6 theta   Free           -0.002             0.000
   7   u_x   Free            0.000             0.000
   8   u_y  Fixed          -45.000           130.555
   9 theta   Free           -0.009             0.000
  10   u_x   Free            0.000             0.000
  11   u_y  Fixed          -15.000            49.287
  12 theta   Free            0.033             0.000


In [35]:
# -------------------------
# Display Element Results
# -------------------------
# Compute and print the element-level results for each member.

print_element(1, u_global, m1, T1, k1, Qf1, disp_in_mm=True, dec=1, rad_dec=6)
print_element(2, u_global, m2, T2, k2, Qf2, disp_in_mm=True, dec=1, rad_dec=6)
print_element(3, u_global, m3, T3, k3, Qf3, disp_in_mm=True, dec=1, rad_dec=6)


E1
u [mm,rad]: [0.0, 0.0, 0.000000, 0.0, 0.0, -0.001954]
v [mm,rad]: [0.0, 0.0, 0.000000, 0.0, 0.0, -0.001954]
q [kN,kN·m]: [0.0, 58.7, 76.5, 0.0, 61.3, -87.0]

E2
u [mm,rad]: [0.0, 0.0, -0.001954, 0.0, -45.0, -0.009059]
v [mm,rad]: [0.0, 0.0, -0.001954, 0.0, -45.0, -0.009059]
q [kN,kN·m]: [0.0, 60.2, 87.0, 0.0, 59.8, -85.7]

E3
u [mm,rad]: [0.0, -45.0, -0.009059, 0.0, -15.0, 0.032563]
v [mm,rad]: [0.0, -45.0, -0.009059, 0.0, -15.0, 0.032563]
q [kN,kN·m]: [0.0, 70.7, 85.7, 0.0, 49.3, 0.0]
